In [ ]:
import cv2
import numpy as np

In [ ]:
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("웹캠을 열수 없습니다.")
while True:
    (ret, frame) = cap.read()
    if not ret:
        print("프레임 오류")
        break

    flip_frame = cv2.flip(frame, 1) # 좌우로 뒤집기
    (heights, width, _) = flip_frame.shape
    (center_x, center_y) = (width // 2, heights // 2)
    roi = flip_frame[center_y - 150:center_y + 150, center_x - 150:center_x + 150]
    cv2.rectangle(flip_frame, (center_x - 150, center_y - 150),(center_x + 150, center_y + 150), (0, 0, 255), 2)

    cv2.imshow("Webcam", flip_frame)

    key = cv2.waitKey(1) & 0xFF
    if key in (ord('c'), ord('C')):
        gray_image = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        gray_image = np.flip(gray_image, axis=1)
        cv2.imwrite('gray_image.png', gray_image)
        gaussian_blur = cv2.GaussianBlur(gray_image, (5, 5), 3)
        (_, otsh_thresh) = cv2.threshold(gaussian_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        cv2.imshow("OTSH", otsh_thresh)
        kernel = np.ones((5, 5), np.uint8)
        erosion = cv2.erode(otsh_thresh, kernel, iterations=5)
        cv2.imshow("EROSION", erosion)
        cv2.imwrite("digit_binary_image.png", erosion)
        img = cv2.imread("digit_binary_image.png", cv2.IMREAD_GRAYSCALE)
        (h, w) = img.shape[:2]
        digit_pixels = np.where(img < 255)
        if digit_pixels[0].size == 0:
            print("No digit pixels found.")
            continue
        margin = 20
        y1 = max(0, int(digit_pixels[0].min()) - margin)
        y2 = min(h, int(digit_pixels[0].max()) + margin + 1)
        x1 = max(0, int(digit_pixels[1].min()) - margin)
        x2 = min(w, int(digit_pixels[1].max()) + margin + 1)

        crop = img[y1:y2, x1:x2]
        cv2.imwrite("crop_image.png", crop)
        cv2.imshow("CROP", crop)
        reversed_image = cv2.bitwise_not(crop)
        final_image = np.zeros((300, 300), dtype=np.uint8)
        (ch, cw) = reversed_image.shape[:2]
        scale = min(260 / max(ch, cw), 1.0)
        if scale != 1.0:
            reversed_image = cv2.resize(reversed_image, (int(cw * scale), int(ch * scale)), interpolation=cv2.INTER_AREA)
            (ch, cw) = reversed_image.shape[:2]
        top = (300 - ch) // 2
        left = (300 - cw) // 2
        final_image[top:top + ch, left:left + cw] = reversed_image
        cv2.imshow("REVERSED_IMAGE", final_image)
        cv2.imwrite("IMAGE_FOR_TEST.png", final_image)


    if key == 27:
        break
cap.release()
cv2.destroyAllWindows()